# 03 — Granularity Correlations

Reproduces paper **Fig. 4**
(`all_models_gran_steerability_PV_scatter.png`,
`all_models_gran_T95_scatter.png`), **Fig. 7**
(`per-model-gran-steerabilitty.png`, sic), **Fig. 8**
(`per-model-gran-T95.png`).

Granularity is the paper's `pl_cross_within` formulation
G_c = mean_ℓ(γ_c(ℓ) / 𝒜_c(ℓ)), implemented in
`grace/diagnostics/granularity.py`.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.markers as mmarkers
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr, pearsonr
from scipy import stats as scipy_stats

from grace.analysis.load_results import load_summary_results
from grace.analysis.t95 import t95
from grace.diagnostics.granularity import granularity

FIG_DIR = Path('Images'); FIG_DIR.mkdir(exist_ok=True)
MODELS = {
    'gemma2': 'google/gemma-2-2b-it',
    'gemma3': 'google/gemma-3-27b-it',
    'llama3': 'meta-llama/Llama-3.3-70B-Instruct',
}
MODEL_SHORT = {'gemma2': 'Gemma2-2B', 'gemma3': 'Gemma3-27B', 'llama3': 'Llama-70B'}
MODEL_COLORS = {'gemma2': 'tab:green', 'gemma3': 'tab:blue', 'llama3': 'tab:orange'}
CONCEPTS = sorted(p.stem for p in Path('concepts/gpt-5/extract').glob('*.json'))

# ── Plotting style ──────────────────────────────────────────────────────────
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 200,
    "savefig.dpi": 300,
    "font.size": 14,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "figure.titleweight": "bold",
})

# Per-concept colour / marker for per-model scatter subplots
_MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', 'p', '*', 'h'] * 2
_CMAP = plt.cm.tab20
CONCEPT_STYLE = {
    c: {'color': _CMAP(i / 20), 'marker': _MARKERS[i]}
    for i, c in enumerate(CONCEPTS)
}
CONCEPT_DISPLAY = {c: c.replace('_', ' ').title() for c in CONCEPTS}

In [ ]:
rows = []
for tag, model_name in MODELS.items():
    for c in CONCEPTS:
        try:
            G, _ = granularity(model_name, c)
        except FileNotFoundError:
            continue
        sums = load_summary_results(model_name, c, method='pv')
        best_utility = max((r['mean_utility'] for r in sums if r.get('mean_utility') is not None), default=None)
        mean_t95, _ = t95(model_name, c, method='pv', mode='unconstrained')
        rows.append({'model': tag, 'concept': c, 'granularity': G,
                     'best_utility': best_utility, 'T95': mean_t95})
df = pd.DataFrame(rows).dropna()
df.head()

## Fig. 4a — Pooled granularity vs. best-found steerability


In [ ]:
sub = df.dropna(subset=['best_utility'])
fig, ax = plt.subplots(figsize=(8, 5.5))
for tag, grp in sub.groupby('model'):
    ax.scatter(grp['granularity'], grp['best_utility'], label=MODEL_SHORT[tag],
               s=65, alpha=0.7, edgecolors='k', linewidths=0.3,
               color=MODEL_COLORS.get(tag))

x_all = sub['granularity'].values
y_all = sub['best_utility'].values
mask = np.isfinite(x_all) & np.isfinite(y_all)
x_all, y_all = x_all[mask], y_all[mask]

if len(x_all) > 2:
    slope, intercept, r, p, _ = scipy_stats.linregress(x_all, y_all)
    xs = np.linspace(x_all.min(), x_all.max(), 100)
    ax.plot(xs, slope * xs + intercept, 'r--', alpha=0.6)
    rho, sp = spearmanr(x_all, y_all)
    ax.set_title(
        f'Granularity vs Best Steerability\n'
        f'Pearson r={r:.2f} (p={p:.3f})  Spearman \u03c1={rho:.2f} (p={sp:.3f})',
        fontsize=15, fontweight='bold',
    )

ax.set_xlabel('Granularity (higher = harder to steer)')
ax.set_ylabel('Best Steerability')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'all_models_gran_steerability_PV_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

## Fig. 4b — Pooled granularity vs. T_95


In [ ]:
sub = df.dropna(subset=['T95'])
if len(sub) >= 3:
    fig, ax = plt.subplots(figsize=(8, 5.5))
    for tag, grp in sub.groupby('model'):
        ax.scatter(grp['granularity'], grp['T95'], label=MODEL_SHORT[tag],
                   s=65, alpha=0.7, edgecolors='k', linewidths=0.3,
                   color=MODEL_COLORS.get(tag))

    x_all = sub['granularity'].values
    y_all = sub['T95'].values
    mask = np.isfinite(x_all) & np.isfinite(y_all)
    x_all, y_all = x_all[mask], y_all[mask]

    if len(x_all) > 2:
        slope, intercept, r, p, _ = scipy_stats.linregress(x_all, y_all)
        xs = np.linspace(x_all.min(), x_all.max(), 100)
        ax.plot(xs, slope * xs + intercept, 'r--', alpha=0.6)
        rho, sp = spearmanr(x_all, y_all)
        ax.set_title(
            f'Granularity vs Trials to 95% of Best\n'
            f'Pearson r={r:.2f} (p={p:.3f})  Spearman \u03c1={rho:.2f} (p={sp:.3f})',
            fontsize=15, fontweight='bold',
        )

    ax.set_xlabel('Granularity (higher = harder to steer)')
    ax.set_ylabel('Average $T_{95}$')
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'all_models_gran_T95_scatter.png', dpi=300, bbox_inches='tight')
    plt.show()

## Fig. 7/8 — Per-model breakouts (3-panel side-by-side)


In [ ]:
def _scatter_subplot(ax, x, y, concepts, title_prefix):
    """Per-model scatter with per-concept colour/marker + regression."""
    for xi, yi, c in zip(x, y, concepts):
        st = CONCEPT_STYLE[c]
        ax.scatter(xi, yi, s=60, alpha=0.7, edgecolors='k', linewidths=0.3,
                   color=st['color'], marker=st['marker'])

    if len(x) > 2:
        mask = np.isfinite(x) & np.isfinite(y)
        xm, ym = x[mask], y[mask]
        slope, intercept, r, p, _ = scipy_stats.linregress(xm, ym)
        xs = np.linspace(xm.min(), xm.max(), 100)
        ax.plot(xs, slope * xs + intercept, 'r--', alpha=0.6)
        rho, sp = spearmanr(xm, ym)
        ax.set_title(
            f'{title_prefix}\n'
            f'Pearson r={r:.2f} (p={p:.3f})\n'
            f'Spearman \u03c1={rho:.2f} (p={sp:.3f})',
            fontsize=11, fontweight='bold',
        )
    else:
        ax.set_title(title_prefix, fontsize=11, fontweight='bold')

def _add_concept_legend(fig):
    """Shared concept legend on the right-hand side of a figure."""
    handles = []
    for c in CONCEPTS:
        st = CONCEPT_STYLE[c]
        h = plt.Line2D([], [], color=st['color'], marker=st['marker'],
                       linestyle='None', markersize=7, markeredgecolor='k',
                       markeredgewidth=0.3, label=CONCEPT_DISPLAY[c])
        handles.append(h)
    fig.legend(handles=handles, loc='center right', fontsize=8,
               frameon=True, borderpad=0.8, handletextpad=0.4,
               labelspacing=0.45)

for ycol, ylabel, title_base, fname in [
    ('best_utility', 'Best Steerability',
     'Per-Model Comparison of Granularity and Best Steerability',
     'per-model-gran-steerabilitty.png'),
    ('T95', 'Average $T_{95}$',
     'Per-Model Comparison of Granularity and $T_{95}$',
     'per-model-gran-T95.png'),
]:
    sub = df.dropna(subset=[ycol])
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    fig.subplots_adjust(right=0.82, wspace=0.32)

    for ax, tag in zip(axes, MODELS):
        s = sub[sub.model == tag]
        if s.empty:
            ax.set_visible(False)
            continue
        _scatter_subplot(
            ax,
            s['granularity'].values,
            s[ycol].values,
            s['concept'].values,
            MODEL_SHORT[tag],
        )
        ax.set_xlabel('Granularity (higher = harder to steer)')
        ax.set_ylabel(ylabel)

    fig.suptitle(title_base, fontsize=15, fontweight='bold', y=0.95)
    _add_concept_legend(fig)
    fig.tight_layout(rect=[0, 0, 0.89, 1])
    fig.savefig(FIG_DIR / fname, dpi=300, bbox_inches='tight')
    plt.show()